In [ ]:
#Importa las librerías necesarias
import requests
import pandas as pd
import numpy as np
from keys import *

#Define ciudad  y pais
city = "New York"
country = "US"

#Realiza la solicitud a la API de OpenWeatherMap con la clave y parámetros
response = requests.get(f'http://api.openweathermap.org/data/2.5/forecast/?q={city},{country}&appid={OWM_key}&units=metric&lang=en')

In [ ]:
# Convierte la respuesta de la API a formato JSON
data = response.json()

# Extrae la lista de pronósticos del JSON
forecast_list = data.get('list', [])

# Crea listas vacías para guardar los datos del clima
times = []
temperatures = []
humidities = []
weather_statuses = []
wind_speeds = []
rain_volumes = []
snow_volumes = []

#Recorre cada pronóstico y extrae los valores
for entry in forecast_list:
    times.append(entry.get('dt_txt', np.nan))
    temperatures.append(entry.get('main', {}).get('temp', np.nan))
    humidities.append(entry.get('main', {}).get('humidity', np.nan))
    weather_statuses.append(entry.get('weather', [{}])[0].get('main', np.nan))
    wind_speeds.append(entry.get('wind', {}).get('speed', np.nan))
    rain_volumes.append(entry.get('rain', {}).get('3h', np.nan))
    snow_volumes.append(entry.get('snow', {}).get('3h', np.nan))

# Crea un DataFrame con toda la información del clima
df = pd.DataFrame({
    'time': times,
    'temperature': temperatures,
    'humidity': humidities,
    'weather_status': weather_statuses,
    'wind_speed': wind_speeds,
    'rain_volume_3h': rain_volumes,
    'snow_volume_3h': snow_volumes
})

#Muestra las primeras filas del DataFrame
print(df.head())

                  time  temperature  humidity weather_status  wind_speed  \
0  2025-11-03 09:00:00        10.91        72         Clouds        1.79   
1  2025-11-03 12:00:00        11.37        72         Clouds        1.55   
2  2025-11-03 15:00:00        15.82        62         Clouds        3.61   
3  2025-11-03 18:00:00        15.18        64         Clouds        4.09   
4  2025-11-03 21:00:00        14.77        61         Clouds        3.62   

   rain_volume_3h  snow_volume_3h  
0             NaN             NaN  
1             NaN             NaN  
2             NaN             NaN  
3             NaN             NaN  
4             NaN             NaN  


In [ ]:
# === Subir DataFrame 'df' a MySQL en la tabla weather_data ===
from sqlalchemy import create_engine
import pymysql
import pandas as pd


user = 'root'
password = 'july0905'
host = '127.0.0.1'
port = 3307
database = 'gans'

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")



In [8]:
# =============================================
# Inserción de datos meteorológicos en MySQL
# =============================================

from sqlalchemy import text  # Necesario para ejecutar SQL crudo

# Normalizamos nombres de columnas según tu tabla weather_data
df_weather = df.rename(columns={
    'time': 'weather_datetime',
    'wind_speed': 'wind',
    'rain_volume_3h': 'rain_qty',
    'snow_volume_3h': 'snow'
}).copy()

# Añadir municipio explícitamente (requerido por la FK)
df_weather['municipality_iso_country'] = "New York,US"

# Convertir fecha a tipo datetime si no lo está
df_weather['weather_datetime'] = pd.to_datetime(df_weather['weather_datetime'])

# (Opcional) Crear la tabla si no existe
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS weather_data (
            id INT AUTO_INCREMENT PRIMARY KEY,
            weather_datetime DATETIME,
            temperature FLOAT,
            humidity INT,
            weather_status VARCHAR(100),
            wind FLOAT,
            rain_qty FLOAT,
            snow FLOAT,
            municipality_iso_country VARCHAR(150)
        );
    """))
    conn.commit()  # Confirma los cambios en la base de datos

# Insertar los datos en la tabla (append)
df_weather.to_sql('weather_data', con=engine, if_exists='append', index=False)

print("✅ DataFrame de clima insertado correctamente en weather_data")



✅ DataFrame de clima insertado correctamente en weather_data


In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# Datos de conexión a MySQL
user = "root"
password = "july0905"
host = "127.0.0.1"
port = 3307
database = "gans"

# Crear conexión al motor de base de datos
engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}")

# Cargar archivo CSV
df = pd.read_csv("worldcities.csv")

# Ver columnas disponibles
print("Columnas disponibles:", df.columns.tolist())

# Seleccionar y limpiar columnas principales (ajustado a worldcities.csv)
df_cities = df[['city', 'iso2', 'population']].dropna()
df_cities = df_cities.rename(columns={'iso2': 'country_code'})

# Crear columna única combinada
df_cities['municipality_iso_country'] = df_cities['city'] + ',' + df_cities['country_code']

# Mostrar las primeras filas
df_cities.head()


Columnas disponibles: ['city', 'city_ascii', 'lat', 'lng', 'country', 'iso2', 'iso3', 'admin_name', 'capital', 'population', 'id']


,city,country_code,population,municipality_iso_country
0,Tokyo,JP,37977000.0,"Tokyo,JP"
1,Jakarta,ID,34540000.0,"Jakarta,ID"
2,Delhi,IN,29617000.0,"Delhi,IN"
3,Mumbai,IN,23355000.0,"Mumbai,IN"
4,Manila,PH,23088000.0,"Manila,PH"


In [ ]:
from sqlalchemy import text

# Crear tabla si no existe
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS city_pop (
            id INT AUTO_INCREMENT PRIMARY KEY,
            city VARCHAR(100),
            country_code VARCHAR(10),
            population FLOAT,
            municipality_iso_country VARCHAR(150) UNIQUE
        );
    """))
    conn.commit()

# Filtrar solo la ciudad de New York
ny = df_cities[df_cities['city'].str.lower() == 'new york']

# Si hay varias filas, tomamos solo la primera
ny = ny.iloc[0:1]

# Insertar solo New York si no existe
with engine.begin() as conn:
    conn.execute(text("""
        INSERT IGNORE INTO city_pop (city, country_code, population, municipality_iso_country)
        VALUES (:city, :country_code, :population, :municipality_iso_country);
    """), ny.to_dict(orient="records"))

print("✅ Ciudad 'New York,US' insertada o ya existente.")



✅ Ciudad 'New York,US' insertada o ya existente.


In [17]:
# Mostrar los primeros registros que acabamos de insertar
pd.read_sql("SELECT * FROM weather_data ORDER BY weather_datetime DESC LIMIT 10;", engine)


,id,weather_datetime,temperature,humidity,weather_status,wind,rain_qty,snow,municipality_iso_country
0,120,2025-11-08 06:00:00,16.80,68,Clouds,6.75,None,None,"New York,US"
1,119,2025-11-08 03:00:00,16.30,74,Clouds,7.80,None,None,"New York,US"
2,118,2025-11-08 00:00:00,15.43,70,Clouds,7.40,None,None,"New York,US"
3,117,2025-11-07 21:00:00,16.07,56,Clouds,8.42,None,None,"New York,US"
4,116,2025-11-07 18:00:00,15.72,48,Clouds,8.42,None,None,"New York,US"
5,115,2025-11-07 15:00:00,12.28,49,Clouds,7.18,None,None,"New York,US"
6,114,2025-11-07 12:00:00,8.94,53,Clouds,5.01,None,None,"New York,US"
7,113,2025-11-07 09:00:00,8.77,51,Clouds,3.23,None,None,"New York,US"
8,112,2025-11-07 06:00:00,8.84,46,Clear,1.79,None,None,"New York,US"
9,111,2025-11-07 03:00:00,9.22,43,Clear,1.42,None,None,"New York,US"
